In [1]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
)



# Ignore all warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "lines.linewidth": 2.5,
})

# Configuration
FORECAST_HORIZONS = [1, 7, 14, 30]
TEST_SIZE_DAYS = 30  # Last 30 days for testing
RANDOM_STATE = 42
LSTM_LOOKBACK = 14

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✓ Imports complete")

✓ Imports complete


<br> <br> <br>

### 1. Creating and Imporing Datasets

In [3]:
# Edges
df_edges_plant = pd.read_csv("../../data/SupplyGraph/Edges/Edges (Plant).csv")
df_edges_product_group = pd.read_csv("../../data/SupplyGraph/Edges/Edges (Product Group).csv")
df_edges_product_subgroup = pd.read_csv("../../data/SupplyGraph/Edges/Edges (Product Sub-Group).csv")
df_edges_storage_location = pd.read_csv("../../data/SupplyGraph/Edges/Edges (Storage Location).csv")

# Nodes
df_nodes_productgroup_and_subgroup = pd.read_csv("../../data/SupplyGraph/Nodes/Node Types (Product Group and Subgroup).csv")
df_nodes_plant_and_storage = pd.read_csv("../../data/SupplyGraph/Nodes/Nodes Type (Plant & Storage).csv")
df_nodes = pd.read_csv("../../data/SupplyGraph/Nodes/Nodes.csv")
# df_nodes_index = pd.read_csv("../data/SupplyGraph/Nodes/NodesIndex.csv")

# Temporal
df_temporal_delivery_to_distributor = pd.read_csv("../../data/SupplyGraph/Temporal Data/Unit/Delivery To distributor.csv")
df_temporal_factory_issue = pd.read_csv("../../data/SupplyGraph/Temporal Data/Unit/Factory Issue.csv")
df_temporal_production = pd.read_csv("../../data/SupplyGraph/Temporal Data/Unit/Production.csv")
df_temporal_sales_order = pd.read_csv("../../data/SupplyGraph/Temporal Data/Unit/Sales Order.csv")

# Pivot the datasets
df_temporal_sales_order['Date'] = pd.to_datetime(df_temporal_sales_order['Date'])
df_temporal_sales_order_pivot = df_temporal_sales_order.melt(id_vars='Date', var_name='Product', value_name='Sales').dropna()
df_temporal_production['Date'] = pd.to_datetime(df_temporal_production['Date'])
df_temporal_production_pivot = df_temporal_production.melt(id_vars='Date', var_name='Product', value_name='Production Quantity').dropna()
df_temporal_factory_issue['Date'] = pd.to_datetime(df_temporal_factory_issue['Date'])
df_temporal_factory_issue_pivot = df_temporal_factory_issue.melt(id_vars='Date', var_name='Product', value_name='Factory Issue').dropna()
df_temporal_delivery_to_distributor['Date'] = pd.to_datetime(df_temporal_delivery_to_distributor['Date'])
df_temporal_delivery_to_distributor_pivot = df_temporal_delivery_to_distributor.melt(id_vars='Date', var_name='Product', value_name='Distributor').dropna()

# Trading Economics Dataset
# daily
df_trading_economics_currency = pd.read_csv("../../data/tradingeconomics/Overview/currency.csv", encoding="utf-16")
df_trading_economics_currency["Date"] = pd.to_datetime(df_trading_economics_currency["Date"], dayfirst=True, errors='coerce')

# daily
df_trading_economics_stock_market = pd.read_csv("../../data/tradingeconomics/Overview/stock_market.csv", encoding="utf-16")
df_trading_economics_stock_market["Date"] = pd.to_datetime(df_trading_economics_stock_market["Date"], dayfirst=True, errors='coerce')

# daily
df_trading_economics_interest_rate = pd.read_csv("../../data/tradingeconomics/Money/Interest_Rate.csv")
df_trading_economics_interest_rate["DateTime"] = pd.to_datetime(df_trading_economics_interest_rate["DateTime"], errors='coerce')
df_trading_economics_interest_rate["LastUpdate"] = pd.to_datetime(df_trading_economics_interest_rate["LastUpdate"], errors='coerce')

# monthly
df_trading_economics_inflation_rate = pd.read_csv("../../data/tradingeconomics/Overview/inflation_rate.csv")
df_trading_economics_inflation_rate["DateTime"] = pd.to_datetime(df_trading_economics_inflation_rate["DateTime"], errors='coerce')
df_trading_economics_inflation_rate["LastUpdate"] = pd.to_datetime(df_trading_economics_inflation_rate["LastUpdate"], errors='coerce')

# monthly
df_trading_economics_balance_of_trade = pd.read_csv("../../data/tradingeconomics/Overview/balance_of_trade.csv")
df_trading_economics_balance_of_trade["DateTime"] = pd.to_datetime(df_trading_economics_balance_of_trade["DateTime"], errors='coerce')
df_trading_economics_balance_of_trade["LastUpdate"] = pd.to_datetime(df_trading_economics_balance_of_trade["LastUpdate"], errors='coerce')

# monthly
df_trading_economics_export = pd.read_csv("../../data/tradingeconomics/Trade/Exports.csv")
df_trading_economics_export["DateTime"] = pd.to_datetime(df_trading_economics_export["DateTime"], errors='coerce')
df_trading_economics_export["LastUpdate"] = pd.to_datetime(df_trading_economics_export["LastUpdate"], errors='coerce')

# monthly
df_trading_economics_import = pd.read_csv("../../data/tradingeconomics/Trade/Imports.csv")
df_trading_economics_import["DateTime"] = pd.to_datetime(df_trading_economics_import["DateTime"], errors='coerce')
df_trading_economics_import["LastUpdate"] = pd.to_datetime(df_trading_economics_import["LastUpdate"], errors='coerce')

In [3]:
# pd.read_csv("../data/SupplyGraph/Temporal Data/Unit/Production.csv")
df_temporal_production_pivot

,Date,Product,Production Quantity
0,2023-01-01,SOS008L02P,1008
1,2023-01-02,SOS008L02P,1014
2,2023-01-03,SOS008L02P,0
3,2023-01-04,SOS008L02P,0
4,2023-01-05,SOS008L02P,0
...,...,...,...
9056,2023-08-05,EEA200G24P,0
9057,2023-08-06,EEA200G24P,0
9058,2023-08-07,EEA200G24P,0
9059,2023-08-08,EEA200G24P,0


<br> <br> <br>


#### 1.1 Create a Holiday Dataset



In [4]:
# @misc{BBHolidays2023,
#   title        = {BB Holidays 2023},
#   author       = {{Bangladesh Bank}},
#   year         = {2023},
#   howpublished = {\url{https://www.scribd.com/document/626649862/BB-Holidays-2023}},
#   note         = {Accessed via Scribd}
# }


holidays_dict = [
    {
        'Date': '2023-02-21',
        'holiday_type': "National Holiday",
        'holiday_name': "Language Mother Day"
    },
    {
        'Date': "2023-03-8",
        'holiday_type': "National Holiday",
        'holiday_name': "Shab e-barat"
    },   
    {
        'Date': "2023-03-17",
        'holiday_type': "National Holiday",
        'holiday_name': "Sheikh Mujibur Rahman's birthday",
    },
    {
        'Date': "2023-03-26",
        'holiday_type': "National Holiday",
        'holiday_name': "Independence Day",
    },
    {
        'Date': "2023-04-14",
        'holiday_type': "National Holiday",
        'holiday_name': "Bengali New Year" 
    },
    {
        'Date': "2023-04-19",
        'holiday_type': "National Holiday",
        'holiday_name': "Shab-e-Qadr"
    },
    {
        'Date': "2023-04-21",
        'holiday_type': "National Holiday",
        'holiday_name': "Jumatul Bidah"
    },
    {
        'Date': "2023-04-22",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid al-Fitr"
    },
    {
        'Date': "2023-04-23",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid al-Fitr"
    },
    {
        'Date': "2023-05-01",
        'holiday_type': "National Holiday",
        'holiday_name': "May Day"
    },
    {
        'Date': "2023-05-04",
        'holiday_type': "National Holiday",
        'holiday_name': "Buddha Purnima"
    },
    {
        'Date': "2023-06-28",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid-ul-Azha"
    },
    {
        'Date': "2023-06-29",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid-ul-Azha"
    },
    {
        'Date': "2023-06-30",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid-ul-Azha"
    },
    {
        'Date': "2023-07-01",
        'holiday_type': "Not a National Holiday",
        'holiday_name': "Bank Holiday"
    },
    {
        'Date': "2023-07-29",
        'holiday_type': "National Holiday",
        'holiday_name': "Ashura"
    },
    {
        'Date': "2023-08-15",
        'holiday_type': "National Holiday",
        'holiday_name': "National Mourning Day"
    },       
]


# Create dataframe from holidays_dict
df_holidays = pd.DataFrame(holidays_dict)
df_holidays['Date'] = pd.to_datetime(df_holidays['Date']).dt.normalize()

# Optional: rename for clarity
df_holidays = df_holidays.rename(columns={
    'holiday_name': 'holiday_name',
    'holiday_type': 'holiday_type'
})

# Display the newly created holidays dataframe
display(df_holidays)


,Date,holiday_type,holiday_name
0,2023-02-21,National Holiday,Language Mother Day
1,2023-03-08,National Holiday,Shab e-barat
2,2023-03-17,National Holiday,Sheikh Mujibur Rahman's birthday
3,2023-03-26,National Holiday,Independence Day
4,2023-04-14,National Holiday,Bengali New Year
5,2023-04-19,National Holiday,Shab-e-Qadr
6,2023-04-21,National Holiday,Jumatul Bidah
7,2023-04-22,National Holiday,Eid al-Fitr
8,2023-04-23,National Holiday,Eid al-Fitr
9,2023-05-01,National Holiday,May Day


In [5]:
print(df_holidays["holiday_name"].value_counts())

holiday_name
Eid-ul-Azha                         3
Eid al-Fitr                         2
Language Mother Day                 1
Shab e-barat                        1
Sheikh Mujibur Rahman's birthday    1
Independence Day                    1
Bengali New Year                    1
Shab-e-Qadr                         1
Jumatul Bidah                       1
May Day                             1
Buddha Purnima                      1
Bank Holiday                        1
Ashura                              1
National Mourning Day               1
Name: count, dtype: int64


<br> <br> <br>

#### 1.2 Create a Weather Dataset

In [6]:
# @dataset{zubair2024high,
#   author       = {Zubair, Md and Ishtiaque Mahee, Md Nafiz and Reza, Khondaker Masfiq and Salim, Md Shahidul and Ahmed, Nasim},
#   title        = {High Volume Real-World Weather Data},
#   year         = {2024},
#   publisher    = {Mendeley Data},
#   version      = {V1},
#   doi          = {10.17632/tbrhznpwg9.1},
#   url          = {https://data.mendeley.com/datasets/tbrhznpwg9/1}
# }


def load_weather_data(data_dir='../data/Weather Datasets/Station Wise Data', begin_date='2023-01-01', end_date='2023-08-09'):
    
    import glob
    
    # Convert date strings to datetime
    begin_date = pd.to_datetime(begin_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()
    
    # Get all CSV files in the directory
    csv_files = glob.glob(os.path.join(data_dir, '*.csv'))
    
    if len(csv_files) == 0:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")
    
    print(f"Found {len(csv_files)} CSV files")
    print(f"Date range: {begin_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    
    all_dfs = []
    
    for file_path in csv_files:
        # Extract city name from filename (remove path and .csv extension)
        city_name = os.path.basename(file_path).replace('.csv', '')
        
        # Read the CSV
        df = pd.read_csv(file_path)
        
        # Add City column
        df['City'] = city_name
        
        # Check if year, month, day columns exist
        year_col = None
        month_col = None
        day_col = None
        
        for col in df.columns:
            col_lower = col.lower()
            if 'year' in col_lower:
                year_col = col
            elif 'month' in col_lower:
                month_col = col
            elif 'day' in col_lower:
                day_col = col
        
        # Create Date column from year, month, day
        if year_col and month_col and day_col:
            df['Date'] = pd.to_datetime(
                df[[year_col, month_col, day_col]].astype(int).rename(
                    columns={year_col: 'year', month_col: 'month', day_col: 'day'}
                )
            ).dt.normalize()
            
            # Drop original year, month, day columns
            df = df.drop(columns=[year_col, month_col, day_col])
            
            # Filter by date range
            df = df[(df['Date'] >= begin_date) & (df['Date'] <= end_date)]
        else:
            # Try to find existing date column
            date_col = None
            for col in df.columns:
                if 'date' in col.lower():
                    date_col = col
                    break
            
            if date_col is not None:
                df['Date'] = pd.to_datetime(df[date_col]).dt.normalize()
                if date_col != 'Date':
                    df = df.drop(columns=[date_col])
                # Filter by date range
                df = df[(df['Date'] >= begin_date) & (df['Date'] <= end_date)]
            else:
                print(f"Warning: No date columns found in {city_name}.csv")
        
        all_dfs.append(df)
        # print(f"  ✓ Loaded {city_name}: {len(df)} rows")
    
    # Concatenate all dataframes
    df_weather = pd.concat(all_dfs, ignore_index=True)
    
    # Reorder columns to put Date and City first
    cols = df_weather.columns.tolist()
    priority_cols = ['Date', 'City']
    other_cols = [c for c in cols if c not in priority_cols]
    df_weather = df_weather[priority_cols + other_cols]
    
    # print(f"\n✓ Combined weather data: {len(df_weather)} total rows")
    # print(f"  Cities: {df_weather['City'].nunique()}")
    # print(f"  Date range in data: {df_weather['Date'].min().strftime('%Y-%m-%d')} to {df_weather['Date'].max().strftime('%Y-%m-%d')}")
    # print(f"  Columns: {list(df_weather.columns)}")
    
    return df_weather


# Load weather data for 2023 with specific date range
df_weather = load_weather_data(
    data_dir='../../data/Weather Datasets/Station Wise Data', 
    begin_date='2023-01-01', 
    end_date='2023-08-09'
)

df_weather.drop(columns="Unnamed: 0", inplace=True)

df_weather

Found 35 CSV files
Date range: 2023-01-01 to 2023-08-09


,Date,City,Station,Rainfall,Sunshine,Humidity,Temperature
0,2023-01-01,Ambaganctg,Ambaganctg,0.0,8.3,65.0,20.4
1,2023-01-02,Ambaganctg,Ambaganctg,0.0,8.9,73.0,19.9
2,2023-01-03,Ambaganctg,Ambaganctg,0.0,7.6,80.0,19.9
3,2023-01-04,Ambaganctg,Ambaganctg,0.0,9.4,75.0,20.4
4,2023-01-05,Ambaganctg,Ambaganctg,0.0,8.0,78.0,19.6
...,...,...,...,...,...,...,...
7730,2023-08-05,Teknaf,Teknaf,0.0,10.7,84.0,28.8
7731,2023-08-06,Teknaf,Teknaf,2.0,7.2,80.0,29.0
7732,2023-08-07,Teknaf,Teknaf,0.0,7.7,83.0,28.7
7733,2023-08-08,Teknaf,Teknaf,36.0,0.0,92.0,27.5


In [7]:
# Filter weather data for Ambaganctg (case-insensitive)
city_name = "Dhaka" # 'Ambaganctg', "Dhaka"
df_weather[df_weather['City'].astype(str).str.strip().str.lower() == city_name.lower()]

,Date,City,Station,Rainfall,Sunshine,Humidity,Temperature
1989,2023-01-01,Dhaka,Dhaka,0.0,4.0,81.0,17.9
1990,2023-01-02,Dhaka,Dhaka,0.0,5.5,71.0,19.1
1991,2023-01-03,Dhaka,Dhaka,0.0,0.0,87.0,16.2
1992,2023-01-04,Dhaka,Dhaka,0.0,0.0,91.0,14.8
1993,2023-01-05,Dhaka,Dhaka,0.0,1.3,86.0,15.2
...,...,...,...,...,...,...,...
2205,2023-08-05,Dhaka,Dhaka,55.0,5.7,81.0,29.4
2206,2023-08-06,Dhaka,Dhaka,5.0,5.0,91.0,27.8
2207,2023-08-07,Dhaka,Dhaka,18.0,4.2,95.0,27.4
2208,2023-08-08,Dhaka,Dhaka,54.0,7.1,91.0,27.3


<br> <br> <br>

#### 1.3 Create an Event Dataset

In [8]:
df_events = pd.read_csv("../../data/BangladeshEvents/event.csv",  parse_dates=["date"])
df_events

,date,category,name,type,scope_or_location,typical_max_temp_C,typical_min_temp_C,typical_rain_mm,weather_basis,impact_on_supply_chain,notes,source_links
0,2023-01-01,school_university,Start of primary & secondary school academic y...,Academic session start,Nationwide,25.4,12.7,7.7,Dhaka Jan climate normals,"School-related demand (uniforms, books, transp...",Academic year in Bangladesh generally runs Jan...,https://dpe.gov.bd/|https://www.unesco.org/en/...
1,2023-01-13,cultural_event,Bishwa Ijtema 2023 – Phase 1 (13–15 Jan),Mass religious congregation (Tablighi Jamaat),Tongi near Dhaka,25.4,12.7,7.7,Dhaka Jan climate normals,Huge temporary population inflow; heavy pressu...,Event spans 3 days; treat as event window in m...,https://www.thedailystar.net/tags/biswa-ijtema
2,2023-01-20,cultural_event,Bishwa Ijtema 2023 – Phase 2 (20–22 Jan),Mass religious congregation,Tongi near Dhaka,25.4,12.7,7.7,Dhaka Jan climate normals,Similar logistical pattern to Phase 1: congest...,Use as second event window close to Dhaka mark...,https://www.thedailystar.net/tags/biswa-ijtema
3,2023-02-14,cultural_event,Pohela Falgun & Valentine's Day,Cultural festival,Urban centres (esp. Dhaka),28.1,15.5,28.9,Dhaka Feb climate normals,"High footfall in parks, campuses, restaurants;...",Good candidate for positive demand shock in ur...,https://www.thedailystar.net/news/bangladesh/n...
4,2023-02-21,national_holiday,Language Martyrs' Day / International Mother L...,Government/National holiday,Nationwide,28.1,15.5,28.9,Dhaka Feb climate normals,Processions and official ceremonies; governmen...,Can model as national_holiday dummy; demand ef...,https://www.officeholidays.com/countries/bangl...
5,2023-03-08,religious_holiday,Shab-e-Barat,Government holiday,Nationwide,32.5,20.4,65.8,Dhaka Mar climate normals,Night prayers and visits to cemeteries; some b...,Mainly timing effect on working hours and urba...,https://calendarific.com/holiday/bangladesh/sh...
6,2023-03-17,national_holiday,Sheikh Mujibur Rahman's Birthday & National Ch...,Government/National holiday,Nationwide,32.5,20.4,65.8,Dhaka Mar climate normals,Government offices closed; ceremonies and gath...,Include as national_holiday dummy.,https://www.officeholidays.com/countries/bangl...
7,2023-03-23,religious_period,Start of Ramadan 1444 (approx.),Religious fasting month,Nationwide,32.5,20.4,65.8,Dhaka Mar climate normals,"Shift of consumption to evening/night (iftar, ...",Model as month-long behavioural regime change ...,https://www.islamic-relief.org.uk/resources/is...
8,2023-03-26,national_holiday,Independence and National Day,Government/National holiday,Nationwide,32.5,20.4,65.8,Dhaka Mar climate normals,Large rallies and parades; government offices ...,Important national_holiday dummy with possible...,https://www.officeholidays.com/countries/bangl...
9,2023-04-14,cultural_holiday,Bengali New Year (Pohela Boishakh),Government holiday / Cultural new year,Nationwide (strongest in cities),33.7,23.6,156.0,Dhaka Apr climate normals,One of the biggest cultural festivals; very hi...,Strong positive demand shock in retail; possib...,https://www.timeanddate.com/holidays/banglades...


<br> <br> <br>

#### 1.4 Create a Currency Exchange Dataset

In [9]:
df_currency_exchange = pd.read_csv("../../data/CurrencyExchange/Bangladeshi_Taka_to_USD.csv",  parse_dates=["Date"])
df_currency_exchange

,Date,BDT_USD
0,2023-01-02 23:58:00,104.7776
1,2023-01-03 23:58:00,101.6990
2,2023-01-04 23:58:00,101.9814
3,2023-01-05 23:58:00,102.0143
4,2023-01-06 23:58:00,102.1778
...,...,...
315,2023-12-25 23:58:00,108.3107
316,2023-12-27 23:58:00,108.0956
317,2023-12-28 23:58:00,108.6817
318,2023-12-29 23:58:00,108.1286


<br> <br> <br>

### 2. Calculate the Coefficient of Variation (CV)

In [10]:
def calculate_product_stats(df, group_col='Product', agg_col='Sales'):
    stats = (
        df.groupby(group_col)
          .agg(
              total_units=(agg_col, 'sum'),
              days_with_activity=(agg_col, lambda x: (x > 0).sum()),
              num_days=(agg_col, 'count'),
              avg_daily=(agg_col, 'mean'),
              std_daily=(agg_col, 'std')
          )
    )
    
    stats['activity_frequency'] = stats['days_with_activity'] / stats['num_days']    
    stats['cv'] = stats['std_daily'] / stats['avg_daily']
    # stats['cv'] = stats['cv'].replace([np.inf, -np.inf], np.nan)
    
    # Reset index to make group_col a regular column
    stats = stats.reset_index()
    
    return stats


# Example usage for all temporal dataframes:

# Sales statistics
sales_stats = calculate_product_stats(df_temporal_sales_order_pivot, group_col='Product', agg_col='Sales')
display(Markdown("**Sales Statistics**"))
display(sales_stats)

**Sales Statistics**

,Product,total_units,days_with_activity,num_days,avg_daily,std_daily,activity_frequency,cv
0,AT5X5K,1.782512e+05,174,221,806.566516,938.839092,0.787330,1.163995
1,ATN01K24P,6.393748e+05,179,221,2893.098569,2701.549761,0.809955,0.933791
2,ATN02K12P,1.370076e+05,177,221,619.943831,615.530467,0.800905,0.992881
3,ATPA1K24P,3.100000e+01,1,221,0.140271,2.085286,0.004525,14.866069
4,ATPPCH5X5K,1.000000e+01,1,221,0.045249,0.672673,0.004525,14.866069
5,ATWWP001K24P,9.223728e+03,159,221,41.736327,52.292598,0.719457,1.252928
6,ATWWP002K12P,3.733000e+03,125,221,16.891401,39.055150,0.565611,2.312132
7,EEA200G24P,5.000000e+00,1,221,0.022624,0.336336,0.004525,14.866069
8,EEA500G12P,1.150000e+02,3,221,0.520362,5.250610,0.013575,10.090302
9,MAC1K25P,8.880000e+02,30,221,4.018100,14.922212,0.135747,3.713749


In [11]:
def classify_speed_segment(stats_df, volume_col='total_units', frequency_col='activity_frequency',  q_high=0.8, q_mid=0.5, freq_threshold=0.5):
    
    df = stats_df.copy()
    
    # Calculate quantiles
    q80 = df[volume_col].quantile(q_high) # e.g. 153888.006 = this value changes based on stats_df
    q50 = df[volume_col].quantile(q_mid)  # e.g. 36836.0001 = this value changes based on stats_df
    print(f"q80 = {q80}")
    print(f"q50 = {q50}")
    print(f"Volume Column is: {volume_col}")
    print(f"if ({volume_col} >= {q80}) and ({frequency_col} >= {freq_threshold}) => `Fast`\nelif {volume_col} >= {q50} => `Medium`\nelse => `Slow`")
    print("\n\n")

    # Classification function
    def classify(row):
        if row[volume_col] >= q80 and row[frequency_col] > freq_threshold: # if total_unit > 153888.006 and "activity_frequency (0.7873)" > 0.4 (0.5=default value) 
            return 'Fast'
        elif row[volume_col] >= q50:
            return 'Medium'
        else:
            return 'Slow'
    
    # Apply classification
    df['speed_segment'] = df.apply(classify, axis=1)
    
    return df



# Classify sales statistics
df_stats_classified = classify_speed_segment(
    sales_stats, 
    volume_col='total_units',
    q_high=0.75,  # Top 25% instead of 20%
    q_mid=0.5,
    freq_threshold=0.4  # Lower frequency threshold
)

df_stats_classified

q80 = 153888.006
q50 = 36836.00000000001
Volume Column is: total_units
if (total_units >= 153888.006) and (activity_frequency >= 0.4) => `Fast`
elif total_units >= 36836.00000000001 => `Medium`
else => `Slow`





,Product,total_units,days_with_activity,num_days,avg_daily,std_daily,activity_frequency,cv,speed_segment
0,AT5X5K,1.782512e+05,174,221,806.566516,938.839092,0.787330,1.163995,Fast
1,ATN01K24P,6.393748e+05,179,221,2893.098569,2701.549761,0.809955,0.933791,Fast
2,ATN02K12P,1.370076e+05,177,221,619.943831,615.530467,0.800905,0.992881,Medium
3,ATPA1K24P,3.100000e+01,1,221,0.140271,2.085286,0.004525,14.866069,Slow
4,ATPPCH5X5K,1.000000e+01,1,221,0.045249,0.672673,0.004525,14.866069,Slow
5,ATWWP001K24P,9.223728e+03,159,221,41.736327,52.292598,0.719457,1.252928,Slow
6,ATWWP002K12P,3.733000e+03,125,221,16.891401,39.055150,0.565611,2.312132,Slow
7,EEA200G24P,5.000000e+00,1,221,0.022624,0.336336,0.004525,14.866069,Slow
8,EEA500G12P,1.150000e+02,3,221,0.520362,5.250610,0.013575,10.090302,Slow
9,MAC1K25P,8.880000e+02,30,221,4.018100,14.922212,0.135747,3.713749,Slow



<br> <br> <br>

#### 2.1 – Identify Product

In [12]:
def get_products_by_speed_segment(df_classified, mover='Fast'):
    """
    Get products filtered by speed segment.
    
    Args:
        df_classified: pandas.DataFrame - classified dataframe with 'speed_segment' column
        mover: str or list - speed segment(s) to filter by
               Can be: 'Fast', 'Medium', 'Slow', 'all'
               Or a list: ['Fast', 'Medium'], ['Fast', 'Slow'], etc.
               
    Returns:
        list: List of unique product names matching the specified segment(s)
    """
    valid_segments = ['Fast', 'Medium', 'Slow']
    
    # Handle 'all' case
    if mover == 'all':
        products = df_classified['Product'].dropna().drop_duplicates().tolist()
        print(f"✓ All products: {len(products)} products")
        return products
    
    # Convert single string to list for uniform processing
    if isinstance(mover, str):
        mover = [mover]
    
    # Validate input
    invalid_segments = [m for m in mover if m not in valid_segments]
    if invalid_segments:
        raise ValueError(f"Invalid segment(s): {invalid_segments}. Valid options: {valid_segments}")
    
    # Filter by speed segment(s)
    mask = df_classified['speed_segment'].isin(mover)
    products = df_classified.loc[mask, 'Product'].dropna().drop_duplicates().tolist()
    
    # Print summary
    print(f"\n{'='*70}")
    print(f"PRODUCTS BY SPEED SEGMENT: {mover}")
    print(f"{'='*70}")
    print(f"Total products: {len(products)}")
    
    # Show breakdown by segment
    for segment in mover:
        segment_count = df_classified[df_classified['speed_segment'] == segment]['Product'].nunique()
        print(f"  - {segment}: {segment_count} products")
    
    # Display product details
    for i, product in enumerate(products, 1):
        stats = df_classified[df_classified['Product'] == product].iloc[0]
        print(f"{i:2d}. {product:15s} | Segment: {stats['speed_segment']:6s} | Volume: {stats['total_units']:>10,.0f} | CV: {stats['cv']:.3f} | Activity: {stats['activity_frequency']:.1%}")
    
    return products


# Example usage:

# Get Fast movers only
# FAST_MOVERS = get_products_by_speed_segment(df_stats_classified, mover='Fast')

# Get Medium movers only
# MEDIUM_MOVERS = get_products_by_speed_segment(df_stats_classified, mover='Medium')

# Get Slow movers only
# SLOW_MOVERS = get_products_by_speed_segment(df_stats_classified, mover='Slow')

# Get Fast and Medium movers
# FAST_AND_MEDIUM = get_products_by_speed_segment(df_stats_classified, mover=['Fast', 'Medium'])

# Get all products
ALL_PRODUCTS = get_products_by_speed_segment(df_stats_classified, mover='all')

✓ All products: 41 products


<br> <br> <br>

### 3. Prepare Base Dataset - Merging Dataframes

In [13]:
# Filter for fast movers
MOVERS = ALL_PRODUCTS

# Filter for movers
df_sales_mover = df_temporal_sales_order[['Date'] + MOVERS].copy()
df_production_mover = df_temporal_production[['Date'] + [p for p in MOVERS if p in df_temporal_production.columns]].copy()
df_factory_issue_mover = df_temporal_factory_issue[['Date'] + [p for p in MOVERS if p in df_temporal_factory_issue.columns]].copy()
df_delivery_mover = df_temporal_delivery_to_distributor[['Date'] + [p for p in MOVERS if p in df_temporal_delivery_to_distributor.columns]].copy()


# Pivot to long format
df_sales_long = df_sales_mover.melt(id_vars='Date', var_name='Product', value_name='Sales')
df_production_long = df_production_mover.melt(id_vars='Date', var_name='Product', value_name='Production')
df_factory_issue_long = df_factory_issue_mover.melt(id_vars='Date', var_name='Product', value_name='Factory_Issue')
df_delivery_long = df_delivery_mover.melt(id_vars='Date', var_name='Product', value_name='Delivery')

# Merge all temporal data
df_base = df_sales_long.merge(df_production_long, on=['Date', 'Product'], how='left')
df_base = df_base.merge(df_factory_issue_long, on=['Date', 'Product'], how='left')
df_base = df_base.merge(df_delivery_long, on=['Date', 'Product'], how='left')

# Fill NaN with 0 (no activity)
df_base = df_base.fillna(0)

# Sort by Product and Date
df_base = df_base.sort_values(['Product', 'Date']).reset_index(drop=True)

# Add Product, Group and Sub-Group
df_meta_filtered = df_nodes_productgroup_and_subgroup.rename(columns={'Node': 'Product'})
df_base = df_base.merge(df_meta_filtered[['Product', 'Group', 'Sub-Group']], on='Product', how='left')

print(f"Base dataset shape: {df_base.shape}")
print(f"\nColumns: {list(df_base.columns)}")
print(f"\nDate range: {df_base['Date'].min()} to {df_base['Date'].max()}")
print(f"Number of products: {df_base['Product'].nunique()}")
print(f"Total records: {len(df_base):,}")

df_base = df_base.drop_duplicates().reset_index(drop=True)
df_base

Base dataset shape: (9282, 8)

Columns: ['Date', 'Product', 'Sales', 'Production', 'Factory_Issue', 'Delivery', 'Group', 'Sub-Group']

Date range: 2023-01-01 00:00:00 to 2023-08-09 00:00:00
Number of products: 41
Total records: 9,282


,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT
...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS


In [14]:
# Prepare dataframes for merging

# 1. Filter weather data for Dhaka only
df_weather_dhaka = df_weather[df_weather['City'].str.lower() == 'dhaka'].copy()
df_weather_dhaka = df_weather_dhaka.drop(columns=['City'])  # Remove City column since it's always Dhaka

# 2. Prepare events dataframe (rename 'date' to 'Date' for consistency)
df_events_clean = df_events.copy()
df_events_clean = df_events_clean.rename(columns={'date': 'Date'})
df_events_clean['Date'] = pd.to_datetime(df_events_clean['Date']).dt.normalize()

# 3. Prepare currency exchange dataframe
df_currency_clean = df_currency_exchange.copy()
df_currency_clean['Date'] = pd.to_datetime(df_currency_clean['Date']).dt.normalize()

# 4. Prepare holidays dataframe
df_holidays_clean = df_holidays.copy()
df_holidays_clean['Date'] = pd.to_datetime(df_holidays_clean['Date']).dt.normalize()

# 5. Ensure df_base Date is normalized
df_base['Date'] = pd.to_datetime(df_base['Date']).dt.normalize()

# Merge all dataframes
print("Merging dataframes...")

# Merge with holidays
df_merged = df_base.merge(df_holidays_clean, on='Date', how='left')
print(f"  ✓ Merged with holidays: {df_merged.shape}")

# Merge with weather (Dhaka only)
df_merged = df_merged.merge(df_weather_dhaka, on='Date', how='left')
print(f"  ✓ Merged with weather (Dhaka): {df_merged.shape}")

# Merge with events
df_merged = df_merged.merge(df_events_clean, on='Date', how='left')
print(f"  ✓ Merged with events: {df_merged.shape}")

# Merge with currency exchange
df_merged = df_merged.merge(df_currency_clean, on='Date', how='left')
print(f"  ✓ Merged with currency exchange: {df_merged.shape}")

# Fill NaN for categorical columns
if 'Holiday' in df_merged.columns:
    df_merged['Holiday'] = df_merged['Holiday'].fillna('no_holiday')
if 'category' in df_merged.columns:
    df_merged['category'] = df_merged['category'].fillna('no_event')

df_merged.drop(columns=["name", "type", "typical_max_temp_C", "typical_min_temp_C", "weather_basis", "impact_on_supply_chain", "notes", "source_links"], inplace=True)

Merging dataframes...
  ✓ Merged with holidays: (9061, 10)
  ✓ Merged with weather (Dhaka): (9061, 15)
  ✓ Merged with events: (9061, 26)
  ✓ Merged with currency exchange: (9061, 27)


In [15]:
df_merged

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,BDT_USD
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,104.7776
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,101.6990
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,101.9814
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,102.0143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,106.5524
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,106.6636
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,107.6734
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,107.7798


## 4. Feature Engineering Pipeline

### 4.1 Temporal Features (Lags and Rolling Windows)

In [16]:
def create_lag_features(df, target_col='Sales', lags=[1, 2, 3, 7, 14, 21, 28]):
    """
    Create lag features for each product
    """
    df_feat = df.copy()
    
    for lag in lags:
        df_feat[f'{target_col}_lag_{lag}'] = df_feat.groupby('Product')[target_col].shift(lag)
    
    return df_feat

def create_rolling_features(df, target_col='Sales', windows=[7, 14, 30]):
    """
    Create rolling window statistics
    """
    df_feat = df.copy()
    
    for window in windows:
        # Rolling mean
        df_feat[f'{target_col}_rolling_mean_{window}'] = (
            df_feat.groupby('Product')[target_col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
        )
        
        # Rolling std
        df_feat[f'{target_col}_rolling_std_{window}'] = (
            df_feat.groupby('Product')[target_col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).std())
        )
        
        # Rolling min/max
        df_feat[f'{target_col}_rolling_min_{window}'] = (
            df_feat.groupby('Product')[target_col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).min())
        )
        
        df_feat[f'{target_col}_rolling_max_{window}'] = (
            df_feat.groupby('Product')[target_col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).max())
        )
    
    return df_feat

# Apply lag features and rolling features for Sales
# print("Creating lag and rolling window features for Sales...")
df_features = create_lag_features(df_merged, target_col='Sales', lags=[1, 2, 3, 7, 14, 21, 28])
# df_features = create_rolling_features(df_features, target_col='Sales', windows=[7, 14, 30])

# Also create features for Production, Factory_Issue, Delivery
# print("Creating lag and rolling window features for Production...")
# df_features = create_lag_features(df_features, target_col='Production', lags=[1, 7, 14])
# df_features = create_rolling_features(df_features, target_col='Production', windows=[1, 7, 14])

# print("Creating lag and rolling window features for Sales...")
# df_features = create_lag_features(df_features, target_col='Factory_Issue', lags=[1, 7, 14])
# df_features = create_rolling_features(df_features, target_col='Factory_Issue', windows=[1, 7, 14])

# print("Creating features for Delivery...")
# df_features = create_lag_features(df_features, target_col='Delivery', lags=[1, 7, 14])
# df_features = create_rolling_features(df_features, target_col='Delivery', windows=[1, 7, 14])

print(f"\n✓ Temporal features created")
print(f"Total features: {df_features.shape[1]}")


✓ Temporal features created
Total features: 26


In [17]:
# df_features.head()

# df_features[["Date","inflation_rate", "balance_of_trade", "exports", "imports"]]



#### Create lag for 'inflation_rate', 'balance_of_trade', 'exports', 'imports'
```python
# Assuming df is your dataframe with 'Date' and features
df_features['Date'] = pd.to_datetime(df_features['Date'])
df_features = df_features.sort_values('Date')

macro_cols = ['inflation_rate', 'balance_of_trade', 'exports', 'imports']

# 2. Apply Strategy: Single Lag + Delta
for col in macro_cols:
    # A. Create the 30-day lag (captures the previous month's value)
    df_features[f'{col}_lag_30'] = df_features[col].shift(30)
    
    # B. Backfill the NaNs! 
    # This prevents losing the first month of data.
    # We assume the value 30 days ago was similar to the first available value.
    df_features[f'{col}_lag_30'] = df_features[f'{col}_lag_30'].bfill()
    
    # C. Create a "Delta" feature (Rate of Change)
    # This is powerful for trees: Is inflation going UP or DOWN?
    df_features[f'{col}_monthly_change'] = df_features[col] - df_features[f'{col}_lag_30']

# Check the data count to ensure no rows were lost
print(f"Total rows: {len(df_features)}")
# print(df[['Date', 'inflation_rate', 'inflation_rate_lag_30', 'inflation_rate_monthly_change']].head(35))
```

In [18]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,BDT_USD,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,104.7776,2642.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,101.6990,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,101.9814,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,102.0143,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,106.5524,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,106.6636,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,107.6734,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,107.7798,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0


In [19]:
def create_supply_chain_features(df):
    """
    Create features capturing supply chain dynamics
    """
    df_feat = df.copy()
    
    # Production-to-sales ratio
    df_feat['production_sales_ratio'] = (df_feat['Production'] / (df_feat['Sales'] + 1e-6))
    
    # Delivery-to-sales ratio
    df_feat['delivery_sales_ratio'] = (df_feat['Delivery'] / (df_feat['Sales'] + 1e-6))
    
    # Factory issue rate (relative to production)
    df_feat['factory_issue_rate'] = (df_feat['Factory_Issue'] / (df_feat['Production'] + 1e-6))
    
    # Cumulative inventory proxy (cumsum of production - sales)
    df_feat['inventory_proxy'] = (
        df_feat.groupby('Product').apply(
            lambda x: (x['Production'] - x['Sales']).cumsum()
        ).reset_index(level=0, drop=True)
    )
    
    # Replace inf with NaN
    df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
    
    return df_feat

print("Creating supply chain interaction features...")
df_features = create_supply_chain_features(df_features)

print(f"✓ Supply chain features created")
print(f"Total features: {df_features.shape[1]}")

Creating supply chain interaction features...
✓ Supply chain features created
Total features: 30


In [20]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,BDT_USD,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,104.7776,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,101.6990,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,101.9814,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,102.0143,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,106.5524,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,106.6636,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,107.6734,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,107.7798,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996


In [21]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,BDT_USD,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,104.7776,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,101.6990,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,101.9814,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,102.0143,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,106.5524,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,106.6636,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,107.6734,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,107.7798,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996


<br> <br> <br>

### Merging daily bases datasets from trading economics

In [22]:
def merge_trading_economics_data(df_features, merging_trading_economics_status=False):
    """
    Merge trading economics data with df_features based on Date column.
    
    Parameters:
    -----------
    df_features : pd.DataFrame
        Base dataframe with a 'Date' column
    merging_trading_economics_status : bool
        If True, merges trading economics dataframes with df_features
        
    Returns:
    --------
    pd.DataFrame
        Merged dataframe with trading economics data if merging_trading_economics_status is True,
        otherwise returns df_features unchanged
    """
    if not merging_trading_economics_status:
        print("Trading economics merging is disabled. Returning original df_features.")
        return df_features.copy()
    
    print("Merging trading economics data...")
    df_merged = df_features.copy()
    
    # Ensure Date column is datetime and normalized
    df_merged['Date'] = pd.to_datetime(df_merged['Date']).dt.normalize()
    
    # 1. Merge with currency data (Open, High, Low, Close)
    df_currency_subset = df_trading_economics_currency[['Date', 'Open', 'High', 'Low', 'Close']].copy()
    df_currency_subset['Date'] = pd.to_datetime(df_currency_subset['Date']).dt.normalize()
    df_currency_subset = df_currency_subset.rename(columns={
        'Open': 'currency_open',
        'High': 'currency_high',
        'Low': 'currency_low',
        'Close': 'currency_close'
    })
    df_merged = df_merged.merge(df_currency_subset, on='Date', how='left')
    print(f"  ✓ Merged with currency data: {df_merged.shape}")
    
    # 2. Merge with interest rate data (Value)
    df_interest_subset = df_trading_economics_interest_rate[['DateTime', 'Value']].copy()
    df_interest_subset['DateTime'] = pd.to_datetime(df_interest_subset['DateTime']).dt.normalize()
    df_interest_subset = df_interest_subset.rename(columns={
        'DateTime': 'Date',
        'Value': 'interest_rate_value'
    })
    df_merged = df_merged.merge(df_interest_subset, on='Date', how='left')
    print(f"  ✓ Merged with interest rate data: {df_merged.shape}")
    
    # 3. Merge with stock market data (Open, High, Low, Close)
    df_stock_subset = df_trading_economics_stock_market[['Date', 'Open', 'High', 'Low', 'Close']].copy()
    df_stock_subset['Date'] = pd.to_datetime(df_stock_subset['Date']).dt.normalize()
    df_stock_subset = df_stock_subset.rename(columns={
        'Open': 'stock_market_open',
        'High': 'stock_market_high',
        'Low': 'stock_market_low',
        'Close': 'stock_market_close'
    })
    df_merged = df_merged.merge(df_stock_subset, on='Date', how='left')
    print(f"  ✓ Merged with stock market data: {df_merged.shape}")
    
    print(f"\n✓ Trading economics data merged successfully!")
    print(f"  Added columns: {[col for col in df_merged.columns if col not in df_features.columns]}")
    
    return df_merged


# Example usage:
df_features.drop(columns=["BDT_USD"], inplace=True)
df_features = merge_trading_economics_data(df_features, merging_trading_economics_status=True)

Merging trading economics data...
  ✓ Merged with currency data: (9061, 33)
  ✓ Merged with interest rate data: (9061, 34)
  ✓ Merged with stock market data: (9061, 38)

✓ Trading economics data merged successfully!
  Added columns: ['currency_open', 'currency_high', 'currency_low', 'currency_close', 'interest_rate_value', 'stock_market_open', 'stock_market_high', 'stock_market_low', 'stock_market_close']


In [23]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_open,stock_market_high,stock_market_low,stock_market_close
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,6177.88,6177.88,6177.88
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,6185.06,6185.06,6185.06
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,6202.63,6202.63,6202.63
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,6193.96,6193.96,6193.96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,6299.66,6299.66,6299.66
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,6315.57,6315.57,6315.57


<br> <br> <br>

### Merging inflation rate from trading economics (montly)

we utilized a constant interpolation method for temporal disaggregation, as aligned with the definitions in the Eurostat ESS Guidelines (2018). Since daily inflation data is not available, assigning the official monthly aggregate to each day is the standard baseline approach to align frequencies without introducing artificial variance or unverified assumptions about daily fluctuations.

In [24]:
df_trading_economics_inflation_rate.head()

,Country,Category,DateTime,Value,Frequency,HistoricalDataSymbol,LastUpdate
0,Bangladesh,Inflation Rate,1994-07-31,5.50,Monthly,BangladeshIR,2016-04-05 14:51:00
1,Bangladesh,Inflation Rate,1994-08-31,7.11,Monthly,BangladeshIR,2016-04-05 14:51:00
2,Bangladesh,Inflation Rate,1994-09-30,6.84,Monthly,BangladeshIR,2016-04-05 14:51:00
3,Bangladesh,Inflation Rate,1994-10-31,8.30,Monthly,BangladeshIR,2016-04-05 14:51:00
4,Bangladesh,Inflation Rate,1994-11-30,7.62,Monthly,BangladeshIR,2016-04-05 14:51:00


In [25]:
def merge_inflation_rate_data(df_features, df_inflation_rate, merge_inflation=True):
    """
    Merge monthly inflation rate data with df_features by mapping each month's inflation rate 
    to every single day within that specific month.
    
    Parameters:
    -----------
    df_features : pd.DataFrame
        Base dataframe with a 'Date' column
    df_inflation_rate : pd.DataFrame
        Inflation rate dataframe with 'DateTime' and 'Value' columns
    merge_inflation : bool
        If True, merges inflation rate data with df_features. If False, returns df_features unchanged (default: True)
        
    Returns:
    --------
    pd.DataFrame
        Merged dataframe with inflation_rate column if merge_inflation is True,
        otherwise returns df_features unchanged
    """
    if not merge_inflation:
        print("Inflation rate merging is disabled. Returning original df_features.")
        return df_features.copy()
    
    print("Merging inflation rate data (monthly to daily mapping)...")
    df_merged = df_features.copy()
    
    # Ensure Date column is datetime and normalized
    df_merged['Date'] = pd.to_datetime(df_merged['Date']).dt.normalize()
    
    # Prepare inflation rate data
    df_inflation_subset = df_inflation_rate[['DateTime', 'Value']].copy()
    df_inflation_subset['DateTime'] = pd.to_datetime(df_inflation_subset['DateTime'])
    
    # Extract year and month from inflation data
    df_inflation_subset['Year'] = df_inflation_subset['DateTime'].dt.year
    df_inflation_subset['Month'] = df_inflation_subset['DateTime'].dt.month
    
    # Rename Value column to be more descriptive
    df_inflation_subset = df_inflation_subset.rename(columns={'Value': 'inflation_rate'})
    
    # Keep only Year, Month, and inflation_rate
    df_inflation_subset = df_inflation_subset[['Year', 'Month', 'inflation_rate']]
    
    # Remove duplicates (keep last entry if multiple entries for same month)
    df_inflation_subset = df_inflation_subset.drop_duplicates(subset=['Year', 'Month'], keep='last')
    
    print(f"  ✓ Prepared inflation data: {len(df_inflation_subset)} unique months")
    
    # Create temporary Year and Month columns in df_merged
    df_merged['Year'] = df_merged['Date'].dt.year
    df_merged['Month'] = df_merged['Date'].dt.month
    
    # Merge inflation rate data based on Year and Month
    df_merged = df_merged.merge(df_inflation_subset, on=['Year', 'Month'], how='left')
    
    # Drop temporary Year and Month columns
    df_merged = df_merged.drop(columns=['Year', 'Month'])
    
    print(f"  ✓ Merged inflation rate data: {df_merged.shape}")
    print(f"  Missing inflation rate values: {df_merged['inflation_rate'].isna().sum()}")
    
    # Show sample of mapping
    if df_merged['inflation_rate'].notna().any():
        sample_month = df_merged[df_merged['inflation_rate'].notna()]['Date'].dt.to_period('M').iloc[0]
        print(f"\n  Sample: All days in {sample_month} mapped to inflation rate: {df_merged[df_merged['Date'].dt.to_period('M') == sample_month]['inflation_rate'].iloc[0]}")
    
    print(f"\n✓ Inflation rate data merged successfully!")
    
    return df_merged


# Example usage:
# Merge inflation rate data
df_features = merge_inflation_rate_data(
    df_features, 
    df_trading_economics_inflation_rate, 
    merge_inflation=True
)

# Without inflation rate data
# df_features = merge_inflation_rate_data(
#     df_features, 
#     df_trading_economics_inflation_rate, 
#     merge_inflation=False
# )

Merging inflation rate data (monthly to daily mapping)...
  ✓ Prepared inflation data: 378 unique months
  ✓ Merged inflation rate data: (9061, 39)
  Missing inflation rate values: 0

  Sample: All days in 2023-01 mapped to inflation rate: 8.57

✓ Inflation rate data merged successfully!


In [26]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_open,stock_market_high,stock_market_low,stock_market_close,inflation_rate
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.57
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,6177.88,6177.88,6177.88,8.57
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,6185.06,6185.06,6185.06,8.57
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,6202.63,6202.63,6202.63,8.57
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,6193.96,6193.96,6193.96,8.57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,6299.66,6299.66,6299.66,9.92
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,6315.57,6315.57,6315.57,9.92


<br> <br> <br>

Merge **df_features** with **df_trading_economics_balance_of_trade** and **df_trading_economics_export** and **df_trading_economics_import**


In [27]:
def merge_trade_economics_data(df_features, df_balance_of_trade, df_export, df_import, merge_trade_data=True):
    """
    Merge balance of trade, export, and import data with df_features based on Date column.
    Uses monthly data mapped to each day within that month.
    
    Parameters:
    -----------
    df_features : pd.DataFrame
        Base dataframe with a 'Date' column
    df_balance_of_trade : pd.DataFrame
        Balance of trade dataframe with 'DateTime' and 'Value' columns
    df_export : pd.DataFrame
        Export dataframe with 'DateTime' and 'Value' columns
    df_import : pd.DataFrame
        Import dataframe with 'DateTime' and 'Value' columns
    merge_trade_data : bool
        If True, merges trade data with df_features. If False, returns df_features unchanged
        
    Returns:
    --------
    pd.DataFrame
        Merged dataframe with trade economics data if merge_trade_data is True,
        otherwise returns df_features unchanged
    """
    if not merge_trade_data:
        print("Trade economics merging is disabled. Returning original df_features.")
        return df_features.copy()
    
    print("Merging trade economics data (monthly to daily mapping)...")
    df_merged = df_features.copy()
    
    # Ensure Date column is datetime and normalized
    df_merged['Date'] = pd.to_datetime(df_merged['Date']).dt.normalize()
    
    # Create temporary Year and Month columns in df_merged
    df_merged['Year'] = df_merged['Date'].dt.year
    df_merged['Month'] = df_merged['Date'].dt.month
    
    # 1. Merge Balance of Trade
    df_balance_subset = df_balance_of_trade[['DateTime', 'Value']].copy()
    df_balance_subset['DateTime'] = pd.to_datetime(df_balance_subset['DateTime'])
    df_balance_subset['Year'] = df_balance_subset['DateTime'].dt.year
    df_balance_subset['Month'] = df_balance_subset['DateTime'].dt.month
    df_balance_subset = df_balance_subset.rename(columns={'Value': 'balance_of_trade'})
    df_balance_subset = df_balance_subset[['Year', 'Month', 'balance_of_trade']]
    df_balance_subset = df_balance_subset.drop_duplicates(subset=['Year', 'Month'], keep='last')
    
    df_merged = df_merged.merge(df_balance_subset, on=['Year', 'Month'], how='left')
    print(f"  ✓ Merged with balance of trade: {df_merged.shape}")
    
    # 2. Merge Exports
    df_export_subset = df_export[['DateTime', 'Value']].copy()
    df_export_subset['DateTime'] = pd.to_datetime(df_export_subset['DateTime'])
    df_export_subset['Year'] = df_export_subset['DateTime'].dt.year
    df_export_subset['Month'] = df_export_subset['DateTime'].dt.month
    df_export_subset = df_export_subset.rename(columns={'Value': 'exports'})
    df_export_subset = df_export_subset[['Year', 'Month', 'exports']]
    df_export_subset = df_export_subset.drop_duplicates(subset=['Year', 'Month'], keep='last')
    
    df_merged = df_merged.merge(df_export_subset, on=['Year', 'Month'], how='left')
    print(f"  ✓ Merged with exports: {df_merged.shape}")
    
    # 3. Merge Imports
    df_import_subset = df_import[['DateTime', 'Value']].copy()
    df_import_subset['DateTime'] = pd.to_datetime(df_import_subset['DateTime'])
    df_import_subset['Year'] = df_import_subset['DateTime'].dt.year
    df_import_subset['Month'] = df_import_subset['DateTime'].dt.month
    df_import_subset = df_import_subset.rename(columns={'Value': 'imports'})
    df_import_subset = df_import_subset[['Year', 'Month', 'imports']]
    df_import_subset = df_import_subset.drop_duplicates(subset=['Year', 'Month'], keep='last')
    
    df_merged = df_merged.merge(df_import_subset, on=['Year', 'Month'], how='left')
    print(f"  ✓ Merged with imports: {df_merged.shape}")
    
    # Drop temporary Year and Month columns
    df_merged = df_merged.drop(columns=['Year', 'Month'])
    
    print(f"\n✓ Trade economics data merged successfully!")
    print(f"  Added columns: balance_of_trade, exports, imports")
    print(f"  Missing balance_of_trade values: {df_merged['balance_of_trade'].isna().sum()}")
    print(f"  Missing exports values: {df_merged['exports'].isna().sum()}")
    print(f"  Missing imports values: {df_merged['imports'].isna().sum()}")
    
    return df_merged


# Merge trade economics data
df_features = merge_trade_economics_data(
    df_features,
    df_trading_economics_balance_of_trade,
    df_trading_economics_export,
    df_trading_economics_import,
    merge_trade_data=True
)

Merging trade economics data (monthly to daily mapping)...
  ✓ Merged with balance of trade: (9061, 42)
  ✓ Merged with exports: (9061, 43)
  ✓ Merged with imports: (9061, 44)

✓ Trade economics data merged successfully!
  Added columns: balance_of_trade, exports, imports
  Missing balance_of_trade values: 0
  Missing exports values: 0
  Missing imports values: 0


In [28]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_open,stock_market_high,stock_market_low,stock_market_close,inflation_rate,balance_of_trade,exports,imports
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.57,-169.1,375.81,544.95
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,6177.88,6177.88,6177.88,8.57,-169.1,375.81,544.95
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,6185.06,6185.06,6185.06,8.57,-169.1,375.81,544.95
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,6202.63,6202.63,6202.63,8.57,-169.1,375.81,544.95
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,6193.96,6193.96,6193.96,8.57,-169.1,375.81,544.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,6299.66,6299.66,6299.66,9.92,-204.4,386.11,590.46
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,6315.57,6315.57,6315.57,9.92,-204.4,386.11,590.46


Based on the code in your notebook, here are the features (columns) selected and merged from each of the specified dataframes into `df_features`. I've listed the original column names from each dataframe and their corresponding renamed feature names in the final dataset:

- **df_trading_economics_currency**:
  - Open → currency_open
  - High → currency_high
  - Low → currency_low
  - Close → currency_close

- **df_trading_economics_stock_market**:
  - Open → stock_market_open
  - High → stock_market_high
  - Low → stock_market_low
  - Close → stock_market_close

- **df_trading_economics_interest_rate**:
  - Value → interest_rate_value

- **df_trading_economics_inflation_rate**:
  - Value → inflation_rate

- **df_trading_economics_balance_of_trade**:
  - Value → balance_of_trade

- **df_trading_economics_export**:
  - Value → exports

- **df_trading_economics_import**:
  - Value → imports

These merges use daily data for currency, stock market, and interest rate (matched on 'Date'), and monthly data for inflation, balance of trade, export, and import (mapped to each day in the month via 'Year' and 'Month'). The 'Date' column is used for alignment, and missing values are handled with left joins.

In [29]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_open,stock_market_high,stock_market_low,stock_market_close,inflation_rate,balance_of_trade,exports,imports
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.57,-169.1,375.81,544.95
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,6177.88,6177.88,6177.88,8.57,-169.1,375.81,544.95
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,6185.06,6185.06,6185.06,8.57,-169.1,375.81,544.95
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,6202.63,6202.63,6202.63,8.57,-169.1,375.81,544.95
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,6193.96,6193.96,6193.96,8.57,-169.1,375.81,544.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,6299.66,6299.66,6299.66,9.92,-204.4,386.11,590.46
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,6315.57,6315.57,6315.57,9.92,-204.4,386.11,590.46


In [30]:
df_features.drop(columns=["stock_market_open", "stock_market_high", "stock_market_low"], inplace=True)

In [31]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_close,inflation_rate,balance_of_trade,exports,imports
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,8.57,-169.1,375.81,544.95
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,8.57,-169.1,375.81,544.95
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,8.57,-169.1,375.81,544.95
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,8.57,-169.1,375.81,544.95
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,8.57,-169.1,375.81,544.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,9.92,-204.4,386.11,590.46
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,9.92,-204.4,386.11,590.46


<br> <br> <br>


In machine learning, raw financial data is often "noisy" and "non-stationary," meaning its statistical properties change over time. Your code snippet isn't just cleaning data; it’s performing **feature engineering** to make market behavior digestible for an algorithm.

Here is the breakdown of what that isolation process does and why it is critical for your pipeline.

---

## What the Process Does

The function transforms a raw price series into a set of normalized signals. Here is the step-by-step logic:

* **Data Sanitization:** By replacing `0` with `NaN` and using `ffill()` (forward fill), it removes "dead" data points without breaking the time continuity. In finance, a 0 price is usually a data error, not a market reality.
* **Log Returns Calculation:** It shifts the focus from **absolute price** to **percentage change**.
* **Momentum Generation:** It extracts a "signal" from the noise by averaging the last 5 days of returns, helping the model see trends rather than just daily fluctuations.

---

## Why You Must Do This in Machine Learning

If you fed raw `currency_close` prices directly into a model, it would likely fail. Here’s why this specific isolation and transformation is necessary:

### 1. Achieving Stationarity

Most ML models (especially linear ones and even many neural networks) assume that the data distribution is **stationary**—meaning the mean and variance don't drift wildly over time.

* **The Problem:** Currency prices trend. A price of 1.05 today is vastly different from 1.50 five years ago.
* **The Solution:** Log returns usually have a mean near zero and a consistent variance, making it much easier for the model to learn patterns that apply across different time periods.

### 2. Preventing Data Leakage and Bias

By isolating the feature and using `shift(1)`, you ensure the model isn't "peeking" into the future. Furthermore, by focusing on returns, you prevent the model from becoming biased toward the absolute scale of the currency. A model trained on a currency at 100.00 won't know what to do when it drops to 50.00 unless it's looking at **relative changes**.

### 3. Normalizing Variance

Financial data is notorious for "Heteroscedasticity" (periods of high volatility followed by low volatility).

* **Log Transformation:** Using  compresses the impact of extreme outliers. It ensures that a move from 10 to 11 is treated with the same mathematical weight as a move from 100 to 110.

### 4. Feature Signal-to-Noise Ratio

Raw prices are mostly noise. By creating the `feature_momentum_5d`, you are providing the model with **domain knowledge**. You are telling the model: *"Don't just look at what happened today; look at the collective sentiment of the business week."* This helps the model identify "momentum" which is a proven factor in quantitative finance.

---

## Summary Table

| Step | ML Reason | Technical Term |
| --- | --- | --- |
| **Replace 0 / ffill** | Prevents math errors ( is undefined). | Data Imputation |
| **Log Returns** | Makes data predictable across time. | Stationarity |
| **Rolling Mean** | Filters out random daily "jitter." | Smoothing / Feature Extraction |
| **Dropping NaNs** | Prevents the model from training on "null" info. | Dimensionality Cleaning |


In [32]:
import pandas as pd
import numpy as np

def clean_and_transform_close_only(df_features):
    """
    Isolates currency_close and transforms it for ML readiness.
    """
    # 1. Keep only the essential column
    # We drop others because '0' values are misleading to ML models
    df = df_features.copy()
    
    # 2. Handle missing/zero values in Close
    # Replacing 0 with NaN then forward-filling (common in finance)
    df['currency_close'] = df['currency_close'].replace(0, np.nan)
    df['currency_close'] = df['currency_close'].ffill()
    
    # 3. Log Returns (Scientific requirement for stationarity)
    # Most journals (e.g. Journal of Forecasting) require returns, not prices.
    # df['feature_log_return'] = np.log(df['currency_close'] / df['currency_close'].shift(1))
    df['currency_log_return'] = np.log(df['currency_close'] / df['currency_close'].shift(1))
    
    # 4. Momentum Feature (5-day moving average of returns)
    # Research shows 'Return Autocorrelation' is a strong short-term signal.
    # df['feature_momentum_5d'] = df['feature_log_return'].rolling(window=5).mean()
    df['currency_momentum_5d'] = df['currency_log_return'].rolling(window=5).mean()
    
    # Drop the first few rows that now have NaNs from the shift/rolling operations
    return df

# Usage:
df_final = clean_and_transform_close_only(df_features)

In [33]:
# Usage:

# df_features_copy = clean_and_transform_close_only(df_features)

df_final.drop(columns=["currency_open", "currency_high", "currency_low", "currency_close"], inplace=True)

df_features = df_final.copy()


In [34]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,interest_rate_value,stock_market_close,inflation_rate,balance_of_trade,exports,imports,currency_log_return,currency_momentum_5d
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,8.57,-169.1,375.81,544.95,NaN,NaN
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,NaN,6177.88,8.57,-169.1,375.81,544.95,NaN,NaN
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,NaN,6185.06,8.57,-169.1,375.81,544.95,-0.024846,NaN
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,NaN,6202.63,8.57,-169.1,375.81,544.95,-0.000484,NaN
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,NaN,6193.96,8.57,-169.1,375.81,544.95,0.001934,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,9.92,-204.4,386.11,590.46,0.000000,0.000883
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,9.92,-204.4,386.11,590.46,0.000000,0.000865
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,NaN,6299.66,9.92,-204.4,386.11,590.46,0.003482,-0.000128
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,NaN,6315.57,9.92,-204.4,386.11,590.46,0.000640,0.000824


<br> <br> <br>


## Feature Scaling & Encoding

In [35]:
def scale_and_encode_features(
    df,
    features_for_scaling=None,
    excluded_features=None,
    no_need_for_scaling_features=None,
    categorical_features=None,
    encoding_method='onehot',   # 'onehot', 'label', 'ordinal', or 'none'
    auto_scale_remaining=True,
    remove_excluded=False,
    scale_encoded=False,
    scaler=None,
):
    """
    Scale numeric features and encode categorical features robustly.

    Returns:
        df_processed (pd.DataFrame), processing_info (dict)
    """
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder

    if not isinstance(df, pd.DataFrame):
        raise ValueError("`df` must be a pandas DataFrame")

    excluded_features = excluded_features or []
    no_need_for_scaling_features = no_need_for_scaling_features or []
    categorical_features = categorical_features or []

    # Work on a copy
    df_processed = df.copy()

    # Build sets
    exclude_set = set(excluded_features) | set(no_need_for_scaling_features) | set(categorical_features)

    # Determine numeric features to scale
    if features_for_scaling is None and auto_scale_remaining:
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
        features_for_scaling = [c for c in numeric_cols if c not in exclude_set]
    elif features_for_scaling is None:
        features_for_scaling = []

    # Validate and filter scaling features
    valid_scaling_features = [c for c in features_for_scaling if c in df.columns]
    non_numeric = [c for c in valid_scaling_features if not pd.api.types.is_numeric_dtype(df[c])]
    if non_numeric:
        # Exclude non-numeric items (safety)
        valid_scaling_features = [c for c in valid_scaling_features if c not in non_numeric]

    # Default scaler
    if scaler is None:
        scaler = MinMaxScaler()

    # Scale numeric features (with fallback coercion)
    if valid_scaling_features:
        try:
            df_processed[valid_scaling_features] = scaler.fit_transform(df_processed[valid_scaling_features])
        except Exception:
            # Coerce to numeric and fill NaN with 0 before re-scaling
            for col in valid_scaling_features:
                df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce').fillna(0)
            df_processed[valid_scaling_features] = scaler.fit_transform(df_processed[valid_scaling_features])

    # Encode categorical features
    valid_categorical_features = [c for c in categorical_features if c in df.columns]
    encoders = {}
    encoded_feature_names = []

    if valid_categorical_features and encoding_method is not None and encoding_method.lower() != 'none':
        method = encoding_method.lower()

        if method == 'onehot':
            # Compatibility for sklearn versions with/without sparse_output
            try:
                ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            except TypeError:
                ohe = OneHotEncoder(sparse=False, handle_unknown='ignore')

            # Fit/transform
            cat_block = df_processed[valid_categorical_features].astype(str)
            encoded_arr = ohe.fit_transform(cat_block)

            # If sparse, convert
            if hasattr(encoded_arr, "toarray"):
                encoded_arr = encoded_arr.toarray()

            # Get feature names safely
            try:
                feature_names = list(ohe.get_feature_names_out(valid_categorical_features))
            except Exception:
                # Fallback name generation
                feature_names = []
                for i, col in enumerate(valid_categorical_features):
                    uniques = pd.Index(cat_block[col].unique()).astype(str)
                    for val in uniques:
                        feature_names.append(f"{col}__{val}")

                # Trim/pad to match columns if necessary
                if encoded_arr.shape[1] != len(feature_names):
                    feature_names = [f"ohe_{i}" for i in range(encoded_arr.shape[1])]

            encoded_df = pd.DataFrame(encoded_arr, columns=feature_names, index=df_processed.index)
            df_processed = df_processed.drop(columns=valid_categorical_features)
            df_processed = pd.concat([df_processed, encoded_df], axis=1)

            encoders['onehot'] = ohe
            encoded_feature_names = feature_names

            # Optionally scale encoded features (if requested)
            if scale_encoded and encoded_feature_names:
                enc_scaler = MinMaxScaler()
                df_processed[encoded_feature_names] = enc_scaler.fit_transform(df_processed[encoded_feature_names])
                encoders['scale_encoded'] = enc_scaler

        elif method == 'label':
            for col in valid_categorical_features:
                le = LabelEncoder()
                df_processed[col] = le.fit_transform(df_processed[col].astype(str))
                encoders[f"label_{col}"] = le

        elif method == 'ordinal':
            oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            df_processed[valid_categorical_features] = oe.fit_transform(df_processed[valid_categorical_features].astype(str))
            encoders['ordinal'] = oe

        else:
            raise ValueError("`encoding_method` must be 'onehot', 'label', 'ordinal', or 'none'")

    # Optionally drop excluded columns from returned dataframe
    if remove_excluded:
        to_drop = [c for c in excluded_features if c in df_processed.columns]
        if to_drop:
            df_processed = df_processed.drop(columns=to_drop)

    # Prepare processing info
    processing_info = {
        'scaler': scaler,
        'encoders': encoders,
        'scaled_features': valid_scaling_features,
        'categorical_features': valid_categorical_features,
        'encoded_feature_names': encoded_feature_names,
        'excluded_features': excluded_features,
        'no_scaling_features': no_need_for_scaling_features,
        'encoding_method': encoding_method,
        'auto_scaled': auto_scale_remaining and (features_for_scaling is None)
    }

    return df_processed, processing_info



excluded_features = ["Date", "Station"]
no_need_for_scaling_features = ["feature_log_return", "feature_momentum_5d", "factory_issue_rate", "delivery_sales_ratio", "production_sales_ratio"]
categorical_features = ['Product', "Group", 'Sub-Group', 'holiday_type', "holiday_name", "category", "scope_or_location"]



df_processed, processing_info = scale_and_encode_features(
    df=df_features,
    excluded_features=excluded_features,
    no_need_for_scaling_features=no_need_for_scaling_features,
    categorical_features=categorical_features,
    encoding_method= 'onehot', # label,
    remove_excluded=False
)


In [36]:
df_processed

,Date,Sales,Production,Factory_Issue,Delivery,Station,Rainfall,Sunshine,Humidity,Temperature,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,interest_rate_value,stock_market_close,inflation_rate,balance_of_trade,exports,imports,currency_log_return,currency_momentum_5d,Product_AT5X5K,Product_ATN01K24P,Product_ATN02K12P,Product_ATPA1K24P,Product_ATPPCH5X5K,Product_ATWWP001K24P,Product_ATWWP002K12P,Product_EEA200G24P,Product_EEA500G12P,Product_MAC1K25P,Product_MAHS025K,Product_MAP1K25P,Product_MAPA1K24P,Product_MAR01K24P,Product_MAR02K12P,Product_MASR025K,Product_POP001L12P,Product_POP001L12P.1,Product_POP002L09P,Product_POP005L04P,Product_POP015K,Product_POP500M24P,Product_POPF01L12P,Product_POV001L24P,Product_POV002L09P,Product_POV005L04P,Product_POV500M24P,Product_SE200G24P,Product_SE500G24P,Product_SO0001L12P,Product_SO0002L09P,Product_SO0005L04P,Product_SO0500M24P,Product_SOP001L12P,Product_SOS001L12P,Product_SOS002L09P,Product_SOS003L04P,Product_SOS005L04P,Product_SOS008L02P,Product_SOS250M48P,Product_SOS500M24P,Group_A,Group_E,Group_M,Group_P,Group_S,Group_nan,Sub-Group_AT,Sub-Group_ATN,Sub-Group_ATPA,Sub-Group_ATPPCH,Sub-Group_ATWWP,Sub-Group_EEA,Sub-Group_MAC,Sub-Group_MAHS,Sub-Group_MAP,Sub-Group_MAPA,Sub-Group_MAR,Sub-Group_MASR,Sub-Group_POP,Sub-Group_POPF,Sub-Group_POV,Sub-Group_SE,Sub-Group_SO,Sub-Group_SOP,Sub-Group_SOS,Sub-Group_nan,holiday_type_National Holiday,holiday_type_Not a National Holiday,holiday_type_nan,holiday_name_Ashura,holiday_name_Bank Holiday,holiday_name_Bengali New Year,holiday_name_Buddha Purnima,holiday_name_Eid al-Fitr,holiday_name_Eid-ul-Azha,holiday_name_Independence Day,holiday_name_Jumatul Bidah,holiday_name_Language Mother Day,holiday_name_May Day,holiday_name_Shab e-barat,holiday_name_Shab-e-Qadr,holiday_name_Sheikh Mujibur Rahman's birthday,holiday_name_nan,category_bank_holiday,category_cultural_event,category_cultural_holiday,category_national_holiday,category_no_event,category_religious_holiday,category_religious_period,category_school_university,category_weather_event,scope_or_location_Chattogram division & south-east,scope_or_location_Nationwide,scope_or_location_Nationwide (strongest in Buddhist communities),scope_or_location_Nationwide (strongest in cities),scope_or_location_Tongi near Dhaka,scope_or_location_Urban centres (esp. Dhaka),scope_or_location_nan
0,2023-01-01,0.090322,0.079538,0.049447,0.038181,Dhaka,0.000000,0.380952,0.777778,0.222222,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,0.621192,NaN,NaN,0.000000,0.600135,0.823701,0.481245,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,2023-01-02,0.036580,0.079538,0.087679,0.125128,Dhaka,0.000000,0.523810,0.619048,0.280193,NaN,0.090322,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,0.625206,NaN,0.000000,0.000000,0.600135,0.823701,0.481245,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2023-01-03,0.080510,0.106050,0.151093,0.119652,Dhaka,0.000000,0.000000,0.873016,0.140097,NaN,0.036580,0.090322,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,0.621892,NaN,0.037881,0.000000,0.600135,0.823701,0.481245,0.127194,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0

In [37]:
output_dir = "../../data/FinalDataset"
os.makedirs(output_dir, exist_ok=True)

# Save df_features (pre-scaling/encoding)
features_path = os.path.join(output_dir, "df_features.csv")
df_features.to_csv(features_path, index=False)

# Save df_processed (scaled/encoded)
processed_path = os.path.join(output_dir, "df_processed.csv")
df_processed.to_csv(processed_path, index=False)

print(f"Saved to: {features_path}")
print(f"Saved to: {processed_path}")

Saved to: ../../data/FinalDataset\df_features.csv
Saved to: ../../data/FinalDataset\df_processed.csv
